# 소마니 논문 전량 재현 (03) — 원문 텍스트에서 끝까지

**목표**: Somani et al. (Communications Medicine, 2024)의 토픽 모델링 파이프라인을
저자 공개 코드(`glp1_reddit-main/`) 기준으로 **가감 없이** 재현한다.
사전계산 임베딩을 쓰지 않고 원문 텍스트 39만 건에서 임베딩 계산부터 직접 수행한다.

## 저자 파이프라인 (topic_model.py 원문 기준)
1. 전처리: URL/`\n`/`&gt;` 제거, 공백 정리, 25자 미만 제외
2. 임베딩: `BAAI/bge-base-en-v1.5` (batch 64)
3. UMAP → 10차원 (n_neighbors=15, min_dist=0.0, cosine, seed 42)
4. HDBSCAN(min_cluster_size=100)으로 군집 수·중심 추정 → 그 중심으로 초기화한 KMeans → BERTopic
5. 토픽 라벨: LLM에 키워드 + 대표 글 5 + 무작위 글 5 프롬프트 → 라벨 3개
6. 주제군: c-TF-IDF → MinMax → UMAP(hellinger) → 실루엣 스캔(3~39) → SpectralClustering
7. 주제군 라벨: LLM
8. (별도 스크립트) 감성 분석: twitter-roberta, 약물명 ±256자 창

## 원안에서 벗어난 곳 — 전부 여기 명시한다
| # | 항목 | 원안 | 본 재현 | 사유 |
|---|---|---|---|---|
| 1 | 연산 장치 | cuda | cuda 있으면 cuda, 없으면 cpu | 하드웨어. 결과 불변, 속도만 차이 |
| 2 | 토픽 라벨 LLM | LLaMA-2 (Ollama 로컬) | Ollama 있으면 원안 그대로, 없으면 Gemini(프롬프트는 원문 그대로) | LLM 라벨은 원래 비결정적. 치환 명시 |
| 3 | 중간 결과 저장 | 없음 | 임베딩/UMAP 결과를 파일로 체크포인트 | 수 시간 작업 보호. 계산 결과 불변 |
| 4 | 릴리스 버그 | 그대로는 실행 불가 | 최소 수정 후 실행, 수정부는 `[수정]` 주석 | 아래 표 참조 |

## 저자 공개 코드의 릴리스 버그 (실행 불가 지점) — 재현 과정의 발견
| 위치 | 문제 |
|---|---|
| `TopicLabeling.__init__` | `find_topic_representations(topic_model, df, ...)` 호출하지만 함수 시그니처는 `(self, n_reps=10)` — TypeError |
| `find_ideal_num_groups(self.c_tf_idf_embed)` | 배열을 `llim` 인자로 전달 — 시그니처 불일치 |
| `plot_topics` | 전역 `topic_model` 참조 (self 누락) |
| `create_topic_table` | `self.topic_labels` 미정의 (라벨러 안에 있음) |
| argparse | `type='str'` — 문자열은 type 인자로 불가 |
| `sentiment_analysis.py` | task명 `sentiment_analysis`(올바른 명칭은 `sentiment-analysis`), `preprocess_dataframe` return 누락, 점수 인덱싱이 라벨 순서 가정 |
| pandas 동작 | `str.replace("http\S+", "")`가 pandas 2.x에선 regex=False 기본이라 **URL 제거가 사실상 미작동** — 원 환경 그대로 두고 실측만 기록 |

## 실행 환경 — 저자 README 그대로
저자 지정: **Python 3.11** + `requirements.txt` 고정 버전. Anaconda Prompt에서:
```
conda create -n glp1_reddit python=3.11 -y
conda activate glp1_reddit
pip install -r "경로\glp1_reddit-main\requirements.txt"
pip install ipykernel openpyxl
python -m ipykernel install --user --name glp1_reddit --display-name "glp1_reddit (py3.11)"
```
그 다음 VS Code 우상단 커널 선택에서 **glp1_reddit (py3.11)** 을 고른다.

## 예상 소요 시간 (CPU 기준, 대략)
| 단계 | 시간 |
|---|---|
| 임베딩 39만 건 | 수 시간 (밤에 걸어두기 권장) |
| UMAP 10차원 | 20~60분 |
| HDBSCAN+KMeans+BERTopic | 30~90분 |
| LLM 라벨 33개 토픽 | 수 분 (API) |
| 감성 분석 전량 | 매우 김 — 마지막 절 참조 |


## 코랩에서 돌리는 경우 — 이 절부터. 로컬(VS Code)이면 다음 절로 건너뛴다

1. 구글 드라이브에 `glp1_repro` 폴더를 만들고 `glp1_db_20231019_edited.pickle`(212.8MB)을 올린다
2. 코랩 메뉴: **런타임 → 런타임 유형 변경 → T4 GPU** 선택
3. 아래 셀 실행 (드라이브 연결 + 저자 고정 버전 설치)
4. 설치 끝나면 **런타임 → 세션 다시 시작** 후, 이 셀을 한 번 더 실행하고 계속 진행
   (numpy 등 기존 로드 버전을 교체하기 때문에 재시작이 한 번 필요하다)

체크포인트가 드라이브에 저장되므로 세션이 끊겨도 이어서 돌릴 수 있다.
GPU 사용량이 바닥나면 다른 계정으로 갈아타되, 드라이브의 `glp1_repro` 폴더를
그 계정과 공유(또는 재업로드)하면 체크포인트째 이어진다.


In [ ]:
# [코랩 전용] 로컬이면 이 셀은 그냥 넘어가도 된다 (자동으로 아무것도 안 함)
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    %pip install -q bertopic==0.16.0 hdbscan==0.8.33 sentence-transformers==2.6.1 transformers==4.39.2 umap-learn==0.5.5 pandas==2.2.2 numpy==1.26.4 scikit-learn==1.4.1.post1 openpyxl
    print("설치 완료 — 처음 설치했다면: 런타임 → 세션 다시 시작 → 이 셀 재실행 후 계속")
else:
    print("로컬 실행 — 이 셀은 할 일 없음")


In [ ]:
# -*- coding: utf-8 -*-
# 0) 환경 확인 — 저자 고정 버전과 대조
import sys
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
print("python:", sys.version)

import numpy, pandas, sklearn
print("numpy:", numpy.__version__, "(저자: 1.26.4)")
print("pandas:", pandas.__version__, "(저자: 2.2.2)")
print("sklearn:", sklearn.__version__, "(저자: 1.4.1.post1)")
import bertopic, umap, hdbscan, sentence_transformers
print("bertopic:", bertopic.__version__, "(저자: 0.16.0)")
print("umap-learn / hdbscan / sentence-transformers OK")
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # [수정1] 원안 'cuda' 고정 → 자동
print("device:", DEVICE)


In [ ]:
# 1) 경로·설정 — 저자 config 원문 그대로 (device만 [수정1])
from pathlib import Path
import pickle, time
import numpy as np
import pandas as pd

# 데이터 위치 자동 탐색: 현재 폴더 → 코랩 드라이브 → 로컬 기본 경로 순
_candidates = [Path.cwd(),
               Path("/content/drive/MyDrive/glp1_repro"),
               Path(r"C:\Users\USER\Documents\Obsidian Vault\60_Portfolio\논문과제-GLP1\repo-skeleton\notebooks")]
NB_DIR = next(p for p in _candidates if (p / "glp1_db_20231019_edited.pickle").exists())
DATA_F = NB_DIR / "glp1_db_20231019_edited.pickle"
OUT = NB_DIR / "full_repro_out"; OUT.mkdir(exist_ok=True)
assert DATA_F.exists(), f"데이터 없음: {DATA_F}"

config = {
    "embeddings": {"name": "BAAI/bge-base-en-v1.5", "device": DEVICE},
    "random_state": 42,
    "topic_hdbscan_params": {"min_cluster_size": 100, "metric": "euclidean",
                             "cluster_selection_method": "eom", "prediction_data": True},
    "topic_umap_params": {"n_neighbors": 15, "n_components": 10, "min_dist": 0.0,
                          "metric": "cosine", "random_state": 42},
    "group_umap_params": {"n_neighbors": 2, "n_components": 3, "min_dist": 0.0,
                          "metric": "hellinger", "spread": 2, "random_state": 42},
}
print("출력 폴더:", OUT)


In [ ]:
# 2) 데이터 로드 + 저자 전처리 원문 그대로
raw = pd.read_pickle(DATA_F)
print("원본 N =", len(raw), "(논문: 391,461)")

df = raw.copy()
# ── 이하 저자 preprocess_dataframe 원문 (pandas 2.x 동작 그대로 둠) ──
df["body"] = df["body"].fillna("")            # [수정] inplace fillna는 2.x에서 경고 — 동작 동일
df["body"] = df["body"].str.replace("http\S+", "")   # 주의: regex=False 기본 → 사실상 미작동 (원 코드 동작 보존)
df["body"] = df["body"].str.replace("\\n", " ")
df["body"] = df["body"].str.replace("&gt;", "")
df["body"] = df["body"].str.replace("\s+", " ", regex=True)
df["body_len"] = df["body"].str.len()
df = df.query("body_len >= 25")
# ── 원문 끝 ──

texts = df["body"].to_list()   # [수정] 원 코드는 df['content'] 참조 — 공개 pickle에는 'content' 없음
print("전처리 후 N =", len(df), f"(25자 미만 {len(raw)-len(df)}건 제외)")
print("저자 수 =", df["author"].nunique())


In [ ]:
# 3) 임베딩 — bge-base-en-v1.5, batch 64 (저자 원문). 최장 단계: 체크포인트 저장 [수정3]
from sentence_transformers import SentenceTransformer

EMB_F = OUT / "embeddings_bge_base.npy"
t0 = time.time()
if EMB_F.exists():
    embeddings = np.load(EMB_F)
    print("체크포인트 로드:", embeddings.shape)
else:
    embedding_model = SentenceTransformer(config["embeddings"]["name"], device=config["embeddings"]["device"])
    embeddings = embedding_model.encode(texts, show_progress_bar=True, batch_size=64)
    np.save(EMB_F, embeddings)
    print(f"임베딩 완료 {embeddings.shape} — {time.time()-t0:,.0f}초")


In [ ]:
# 4) UMAP → 10차원 (저자 파라미터 그대로) — 체크포인트 [수정3]
from umap import UMAP

U_F = OUT / "umap10.npy"
t0 = time.time()
if U_F.exists():
    u = np.load(U_F)
    print("체크포인트 로드:", u.shape)
else:
    umap_model = UMAP(**config["topic_umap_params"])
    u = umap_model.fit_transform(embeddings)
    np.save(U_F, u)
    print(f"UMAP 완료 {u.shape} — {time.time()-t0:,.0f}초")


In [ ]:
# 5) HDBSCAN 초기 군집 → 중심점 → KMeans → BERTopic (저자 로직 그대로)
from hdbscan import HDBSCAN
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired

t0 = time.time()
hdbscan_model = HDBSCAN(**config["topic_hdbscan_params"])
clusters = np.array(hdbscan_model.fit_predict(u))
n_clusters = clusters.max() + 1
print(f"HDBSCAN 군집 수 = {n_clusters} (노이즈 {int((clusters==-1).sum())}건) — 논문 33개와 대조")

centroids = np.empty((n_clusters, u.shape[1]))
for i in range(n_clusters):
    centroids[i] = u[clusters == i].mean(axis=0)

kmeans_model = KMeans(n_clusters=n_clusters, random_state=42, init=centroids)
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))
embedding_model = SentenceTransformer(config["embeddings"]["name"], device=DEVICE)

topic_model = BERTopic(vectorizer_model=vectorizer_model,
                       embedding_model=embedding_model,
                       umap_model=UMAP(**config["topic_umap_params"]),
                       hdbscan_model=kmeans_model,
                       representation_model=KeyBERTInspired(),
                       verbose=True)
topics, _ = topic_model.fit_transform(texts, embeddings)
topics = np.array(topics)

with open(OUT / "topics.pickle", "wb") as fh:
    pickle.dump(topics, fh)
topic_model.save(str(OUT / "topic_model"), serialization="safetensors", save_ctfidf=True)
print(f"BERTopic 완료 — {time.time()-t0:,.0f}초 · 최종 토픽 수 = {topics.max()+1}")


In [ ]:
# 6) 토픽 크기 확인 — 논문 Figure/topic_table.docx 와 대조할 1차 수치
info = topic_model.get_topic_info()
print(info[["Topic", "Count", "Name"]].to_string())
info.to_excel(OUT / "topic_info.xlsx")


In [ ]:
# 7) 토픽별 대표 글 5 + 무작위 글 5 (저자 static 메서드 로직 — [수정2] 시그니처 정리)
from sklearn.metrics.pairwise import cosine_distances
from tqdm import tqdm

def find_representative_docs_per_topic(df, topics, embeddings, n_reps=5):
    n_topics = topics.max() + 1
    emb_dim = embeddings.shape[1]
    samples_in = [np.where(topics == i)[0] for i in range(n_topics)]
    cents = np.array([embeddings[s].mean(axis=0) for s in samples_in])
    reps = []
    for cent, s in tqdm(zip(cents, samples_in), total=n_topics):
        d = cosine_distances(embeddings[s], cent.reshape(1, emb_dim)).flatten()
        reps.append(df.iloc[s[np.argsort(d)[:n_reps]]]["body"].values.tolist())
    return reps

def find_random_docs_per_topic(df, topics, n_reps=5):
    out = []
    for t in range(topics.max() + 1):
        idx = np.where(topics == t)[0]
        s = df.iloc[idx]["body"]
        out.append((s if len(idx) < n_reps else s.sample(n=n_reps, random_state=42)).values.tolist())
    return out

rep_docs = find_representative_docs_per_topic(df, topics, embeddings, 10)
rand_docs = find_random_docs_per_topic(df, topics, 10)
prompt_docs = [i[:5] + j[:5] for i, j in zip(rep_docs, rand_docs)]
print("대표/무작위 글 추출 완료")


In [ ]:
# 8) 토픽 라벨링 프롬프트 — 저자 원문 문구 그대로
def prepare_prompt(topic_model, prompt_docs, topic_of_interest):
    prompt = ('You are a honest, scientific chatbot that helps me, a Cardiologist, create unique, '
              'diverse labels for a topic based on representative discussions and keywords. '
              'Do not be creative or loquacious. Please present the topic label in a short and direct manner.\n\n')
    prompt += "I have a topic that is described by the following keywords:\n'"
    prompt += "', '".join(topic_model.get_topic_info(topic_of_interest)["Representation"][0])
    prompt += ("' .\n\nIn this topic, the following documents are a small but representative subset "
               "of all other documents in the topic:\n\n")
    prompt += "\n\n".join(prompt_docs[topic_of_interest])
    prompt += "\n\nBased on the information above, can you create three short, direct labels without descriptions for this topic?"
    return prompt

prompts = {t: prepare_prompt(topic_model, prompt_docs, t) for t in range(topics.max() + 1)}
print("프롬프트", len(prompts), "개 준비")


In [ ]:
# 9) LLM 라벨링 — 1순위 Ollama llama2(원안), 없으면 Gemini(프롬프트 동일, 치환 명시)
import os, json as _json, requests as _rq

def llm_ollama(prompt):
    r = _rq.post("http://localhost:11434/api/generate",
                 json={"model": "llama2", "prompt": prompt, "stream": False}, timeout=300)
    r.raise_for_status()
    return r.json()["response"]

def _gemini_key():
    k = os.environ.get("GEMINI_API_KEY")
    if k:
        return k
    sec = NB_DIR.parent / "app" / ".streamlit" / "secrets.toml"
    if sec.exists():
        for line in sec.read_text(encoding="utf-8").splitlines():
            if line.strip().startswith("GEMINI_API_KEY"):
                return line.split("=", 1)[1].strip().strip('"')
    return None

def llm_gemini(prompt):
    r = _rq.post("https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent",
                 headers={"x-goog-api-key": _gemini_key(), "Content-Type": "application/json"},
                 json={"contents": [{"parts": [{"text": prompt}]}],
                       "generationConfig": {"temperature": 0.1, "maxOutputTokens": 512,
                                            "thinkingConfig": {"thinkingBudget": 0}}},
                 timeout=60)
    r.raise_for_status()
    return r.json()["candidates"][0]["content"]["parts"][0]["text"]

try:
    llm_ollama("Say OK.")
    call_llm, LLM_USED = llm_ollama, "llama2 (Ollama) — 원안 그대로"
except Exception:
    assert _gemini_key(), "Ollama도 Gemini 키도 없음 — 하나는 필요"
    call_llm, LLM_USED = llm_gemini, "gemini-2.5-flash — 원안 llama2 치환 (문서에 명시할 것)"
print("사용 LLM:", LLM_USED)

REP_F = OUT / "topic_representations.json"
representations = _json.loads(REP_F.read_text(encoding="utf-8")) if REP_F.exists() else {}
for t in tqdm(range(topics.max() + 1)):
    if str(t) in representations:
        continue
    representations[str(t)] = call_llm(prompts[t])
    REP_F.write_text(_json.dumps(representations, ensure_ascii=False, indent=1), encoding="utf-8")
print("토픽 라벨 응답", len(representations), "개")


In [ ]:
# 10) 라벨 추출 — 저자 정규식 원문 + 실패 시 줄 단위 폴백 [수정]
import re
pattern = r'(?<=\d(.|:)\s)(.*?)(?=(\\n)+(?:Label )?\d+(.|:)|(\n|\Z))'

def extract_labels(text):
    got = [m[1].replace('"', '').strip() for m in re.findall(pattern, text)]
    got = [g for g in got if g]
    if not got:   # [수정] LLM 출력 형식이 다르면 번호 매긴 줄을 직접 수집
        got = [re.sub(r'^\s*\d+[.):]\s*', '', ln).strip().strip('"')
               for ln in text.splitlines() if re.match(r'^\s*\d+[.):]\s*\S', ln)]
    return got[:3]

topic_labels = ['. '.join(extract_labels(representations[str(t)])) or f"(추출 실패 topic {t})"
                for t in range(topics.max() + 1)]
for t, lab in enumerate(topic_labels):
    print(f"{t:>3} | {lab}")


In [ ]:
# 11) 주제군 묶기 — c-TF-IDF → MinMax → UMAP(hellinger) → 실루엣 스캔 → SpectralClustering (저자 로직)
from sklearn.preprocessing import MinMaxScaler as mms
from sklearn.cluster import SpectralClustering
from sklearn.metrics import silhouette_score

c_tf_idf_mms = mms().fit_transform(topic_model.c_tf_idf_.toarray())   # [수정] 0.16에선 c_tf_idf_
c_tf_idf_vis = UMAP(n_neighbors=2, n_components=2, metric="hellinger",
                    random_state=42).fit_transform(c_tf_idf_mms)
c_tf_idf_embed = UMAP(**config["group_umap_params"]).fit_transform(c_tf_idf_mms)

ss, cluster_arr = [], np.arange(3, 40, 2)
for n in cluster_arr:   # [수정2] 원 코드는 배열을 llim으로 넘기는 버그 — 로직대로 재구성
    cl = SpectralClustering(n_clusters=int(n), random_state=42, n_components=2).fit_predict(c_tf_idf_embed)
    ss.append(silhouette_score(c_tf_idf_embed, cl))
ideal_n = int(cluster_arr[int(np.argmax(ss))])
print(f"top silhouette {max(ss):.3f} @ n_clusters={ideal_n}")

groups = SpectralClustering(n_clusters=ideal_n, random_state=42).fit_predict(c_tf_idf_embed) + 1
print("주제군 분포:", np.bincount(groups)[1:])


In [ ]:
# 12) 주제군 라벨 — 저자 프롬프트 원문 그대로
group_labels_combined = {}
for g in range(groups.min(), groups.max() + 1):
    group_labels_combined[g] = "".join(topic_labels[i].replace('"', '') + "\n"
                                       for i in np.where(groups == g)[0])

group_labels = {}
for g, combined in tqdm(group_labels_combined.items()):
    prompt = ('You are a honest, scientific chatbot that helps me, a Cardiologist, create a single '
              'representative label for a group that represents a series of topics based on topic labels. '
              'Each topic label is separated by a new line character. Do not be creative or loquacious. '
              'Please present the group label in a short and direct manner.\n\n'
              "I have a group that is described by the following topic labels:\n'" + combined +
              "\n\nBased on the information above, can you create the one, best, direct label for this topic "
              "in the following format?\nGroup Label: <group_label>")
    resp = call_llm(prompt)
    m = re.findall(r"(?<=Group Label: )(.*)", resp)
    group_labels[g] = (m[0] if m else resp.strip().splitlines()[-1]).strip().strip('"')  # [수정] 형식 이탈 폴백

for g, lab in group_labels.items():
    print(f"군 {g}: {lab}")


In [ ]:
# 13) 그림 — 저자 plot_topics 로직 ([수정2] 전역 참조 정리)
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_style("white")
plt.figure(figsize=(12, 6))
sizes = topic_model.get_topic_info().set_index("Topic").loc[range(topics.max() + 1), "Count"].values
ax = sns.scatterplot(x=c_tf_idf_vis[:, 0], y=c_tf_idf_vis[:, 1], size=sizes, hue=groups,
                     sizes=(100, 5000), alpha=0.5, palette="tab20", legend=True, edgecolor="k")
h, l = ax.get_legend_handles_labels()
ng = groups.max() - groups.min() + 1
legend = plt.legend(h[:ng], list(group_labels.values()), bbox_to_anchor=(1.02, 1),
                    loc="upper left", fontsize=9)
ax.set_title("Topics, Grouped by Similarity of Content", fontsize=16, pad=10)
ax.set_xlabel("Feature 1"); ax.set_ylabel("Feature 2")
ax.set_xticklabels([]); ax.set_yticklabels([])
plt.savefig(OUT / "figure_groups.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# 14) 토픽 표 — 저자 create_topic_table 로직 → 논문 topic_table.docx 와 대조
rep1 = find_representative_docs_per_topic(df, topics, embeddings, 1)
topic_table = pd.DataFrame(index=np.arange(1, topics.max() + 2, dtype=int),
                           columns=["Discussions (#)", "Group", "Topic Label", "Representative Post"])
for g in range(1, groups.max() + 1):
    for i in np.where(groups == g)[0]:
        topic_table.loc[i + 1, "Topic Label"] = topic_labels[i]
        topic_table.loc[i + 1, "Group"] = g
        topic_table.loc[i + 1, "Representative Post"] = rep1[i][0]
        topic_table.loc[i + 1, "Discussions (#)"] = int((topics == i).sum())
topic_table.to_excel(OUT / "topic_table_repro.xlsx")
print(topic_table[["Discussions (#)", "Group", "Topic Label"]].to_string())
print("\n→ 저자 배포본 topic_table.docx 와 나란히 놓고 토픽 대응을 확인할 것")


## 감성 분석 (저자 sentiment_analysis.py) — 선택 실행

저자 방식: 각 글에서 약물명이 나오는 위치마다 앞뒤 256자 창을 잘라
`cardiffnlp/twitter-roberta-base-sentiment-latest`로 부정/중립/긍정 점수를 내고 평균한다.

⚠️ **전량(39만 건)은 CPU로 수일 걸릴 수 있다.** 아래 셀은 `SENT_LIMIT`으로 실행 범위를 조절한다
— `None`이면 전량(가감 없음), 숫자면 그 건수까지만(부분 실행임을 결과에 명시할 것).


In [ ]:
# 15) 감성 분석 — 저자 로직, [수정] task명·점수 인덱싱·return 누락 교정
SENT_LIMIT = 2000   # 예행용. 전량 실행 시 None (매우 오래 걸림 — 밤에 걸어둘 것)

from transformers import pipeline as hf_pipeline

search_strings = ['semaglutide', 'rybelsus', 'wegovy', 'ozempic', 'retatrutide',
                  'dulaglutide', 'trulicity', 'tirzepatide', 'mounjaro',
                  'liraglutide', 'saxenda', 'exenatide', 'bydureon', 'byetta',
                  'lixisenatide', 'adlyxin']
sent_model = hf_pipeline("sentiment-analysis",   # [수정] 원문 'sentiment_analysis'는 존재하지 않는 task명
                         model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                         tokenizer="cardiffnlp/twitter-roberta-base-sentiment-latest",
                         top_k=None, device=0 if DEVICE == "cuda" else -1)
drug_re = re.compile("|".join(search_strings))

def find_string_sentiment(text):
    idxs = [m.start() for m in drug_re.finditer(text.lower())]
    if not idxs:
        return np.array([np.nan] * 3)
    rows = []
    for ix in idxs:
        window = text[max(0, ix - 256): min(len(text), ix + 256)]
        res = sent_model(window, truncation=True, max_length=512)[0]
        by = {d["label"]: d["score"] for d in res}   # [수정] 라벨명으로 매핑 (원문은 순서 가정)
        rows.append([by["negative"], by["neutral"], by["positive"]])
    return np.mean(rows, axis=0)

target = df if SENT_LIMIT is None else df.head(SENT_LIMIT)
out_rows = [find_string_sentiment(t) for t in tqdm(target["body"].tolist())]
sentiments = pd.DataFrame(out_rows, columns=["negative", "neutral", "positive"], index=target.index)
valid = sentiments.dropna()
sentiments["net"] = np.nan
sentiments.loc[valid.index, "net"] = np.argmax(valid.values, axis=1)
merged = target.join(sentiments)
merged.to_pickle(OUT / "sentiment_partial.pickle")
scope = "전량" if SENT_LIMIT is None else f"선두 {SENT_LIMIT}건 (부분 실행 — 보고 시 명시)"
print(f"범위: {scope} · 약물명 포함 {len(valid)}건")
print(valid[["negative", "neutral", "positive"]].mean().round(3))


## 결과 기록 — 실행 후 직접 채울 것

| 항목 | 논문/저자 배포본 | 본 재현 실측 |
|---|---|---|
| 원본 N | 391,461 | |
| 전처리 후 N (≥25자) | (논문 미기재) | |
| HDBSCAN 초기 군집 수 | — | |
| 최종 토픽 수 | 33 | |
| 주제군 수 (실루엣 선택) | (topic_table.docx 와 대조) | |
| 사용 LLM | LLaMA-2 | |
| 임베딩 소요 시간 | — | |

**정직성 원칙**: 수치가 논문과 다르게 나오면 그대로 기록한다. UMAP·HDBSCAN·LLM은
시드를 고정해도 라이브러리 버전·하드웨어에 따라 결과가 흔들릴 수 있고,
"완전 일치"가 아니라 **"같은 구조가 재현되는가"**(토픽 수 규모, 주요 주제 대응)가 재현의 기준이다.
차이가 나면 그 차이 자체가 보고서의 내용이 된다.
